# Big Data Analytics — Assignment 03
> Author : Badr TAJINI - Big Data Analytics - ESIEE 2025-2026

**Chapter 5 :** Graphs (PageRank/PPR)   
**Chapter 6 :** Spam classification (SGD) in PySpark

**Tools :** Spark or PySpark.   
**Advice:** Keep evidence and reproducibility.


## 0. Bootstrap

In [1]:
# write some code here
# - create SparkSession('BDA-A03') with UTC timezone
# - print Spark/PySpark/Python versions
# - set spark.sql.shuffle.partitions for local runs
import sys
import os
import platform
import pyspark
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BDA-Assignment03")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4") 
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"PySpark version: {pyspark.__version__}")
print(f"Python version: {sys.version.split()[0]}")
print(f"OS: {platform.system()} {platform.release()}")
print(f"Session timezone: {spark.conf.get('spark.sql.session.timeZone')}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/18 10:20:39 WARN Utils: Your hostname, Elliot, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/11/18 10:20:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/18 10:20:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1
PySpark version: 4.0.1
Python version: 3.10.19
OS: Linux 6.6.87.2-microsoft-standard-WSL2
Session timezone: UTC
Shuffle partitions: 4


## 1. Dataset acquisition

In [2]:
# write some code here
# - ensure data/p2p-Gnutella08-adj.txt exists (convert from SNAP edgelist if needed)
# - ensure spam.train.* and spam.test.qrels.txt exist (download + bunzip2)
# - quick sanity checks on file sizes and line counts
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUTS_DIR = BASE_DIR / "outputs"
PROOF_DIR = BASE_DIR / "proof"

for directory in (OUTPUTS_DIR, PROOF_DIR):
    directory.mkdir(exist_ok=True)

# Vérification Graph
graph_path = DATA_DIR / "p2p-Gnutella08-adj.txt"
if graph_path.exists():
    print(f"Graph found: {graph_path.name}")
else:
    print(f"Graph MISSING at {graph_path}")

spam_dir = DATA_DIR / "spam"
if spam_dir.exists():
    print(f"Spam dir found. Contents:")
    for item in sorted(spam_dir.glob("*")):
        print(f"   - {item.name}")
else:
    print(f"Spam dir MISSING at {spam_dir}")

Graph found: p2p-Gnutella08-adj.txt
Spam dir found. Contents:
   - spam.test.qrels.txt
   - spam.test.qrels.txt.bz2
   - spam.train.britney.txt
   - spam.train.britney.txt.bz2
   - spam.train.group_x.txt
   - spam.train.group_x.txt.bz2
   - spam.train.group_y.txt
   - spam.train.group_y.txt.bz2


## 2. Helpers

In [10]:
# write some code here
# - parse adjacency-list line 'u v1 v2 ...' to (u, [v1, v2, ...])
# - utility for top-k without collect: use takeOrdered on (rank, node) with key
# - formatting helpers to save top-20 CSVs
import csv
from typing import Tuple, List, Optional

#Graph Parsing Utility 
def parse_adj_line(line: str) -> Optional[Tuple[str, List[str]]]:
    """ 'u v1 v2 ...' -> ('u', ['v1', 'v2', ...])    """
    parts = line.strip().split()
    if not parts:
        return None
    return (parts[0], parts[1:])

#Efficient Top-K Utility 
def get_top_k(rdd, k: int) -> List[Tuple[str, float]]:
    """ utility for top-k without collect: use takeOrdered on (rank, node) with key """
    return rdd.takeOrdered(k, key=lambda x: -x[1])

#CSV Output Helper 
def save_results_csv(data: List[Tuple[str, float]], filepath: str, columns: List[str] = ["node_id", "score"]):
    """ formatting helpers to save top-20 CSVs """
    try:
        with open(filepath, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(columns)  
            writer.writerows(data)   
        print(f"[IO] Saved {len(data)} rows to: {filepath}")
    except IOError as e:
        print(f"[ERROR] Could not save CSV to {filepath}: {e}")

def clear_output(path):
    """Supprime le dossier de sortie s'il existe déjà pour éviter les erreurs Spark."""
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Deleted old output: {path}")

## 3. Part A — PageRank

In [5]:
# write some code here
# - parameters: alpha=0.85, iterations, partitions
# - initialize ranks uniformly; build adjacency RDD partitioned by key
# - iterative loop: contributions + missing mass redistribution
# - compute top-20 without collect; write outputs/pagerank_top20.csv
# - save any DF stage plan to proof/plan_pr.txt
import time
from operator import add

sc = spark.sparkContext

ALPHA = 0.85
ITERATIONS = 10
PARTITIONS = 4

print("Starting PageRank...")
start_time = time.time()

raw_lines = sc.textFile(str(graph_path))
links = raw_lines.map(parse_adj_line) \
                 .filter(lambda x: x is not None) \
                 .partitionBy(PARTITIONS) \
                 .cache()

all_nodes = raw_lines.flatMap(lambda line: line.split()).distinct()
N = all_nodes.count()
print(f"Graph loaded. Nodes: {N}")

ranks = all_nodes.map(lambda n: (n, 1.0 / N)).partitionBy(PARTITIONS).cache()

with open(PROOF_DIR / "plan_pr.txt", "w") as f:
    f.write(links.toDebugString().decode("utf-8"))

#Iterative loop
base_teleport = (1.0 - ALPHA) / N

for i in range(ITERATIONS):
    dangling_mass = ranks.subtractByKey(links).map(lambda x: x[1]).sum()
    dangling_dist = (ALPHA * dangling_mass) / N
    total_add = base_teleport + dangling_dist

    contribs = links.join(ranks).flatMap(
        lambda kv: [(target, kv[1][1] / len(kv[1][0])) for target in kv[1][0]]
    )

    new_ranks = (
        contribs.reduceByKey(add)
        .mapValues(lambda x: x * ALPHA)
        .rightOuterJoin(ranks)
        .mapValues(lambda x: (x[0] or 0.0) + total_add)
        .partitionBy(PARTITIONS)
    )

    new_ranks.cache()
    ranks.unpersist()
    ranks = new_ranks
    print(f"Iteration {i+1} completed")

#Save results
top_20 = get_top_k(ranks, 20)

output_file = OUTPUTS_DIR / "pagerank_top20.csv"
save_results_csv(top_20, str(output_file), ["node_id", "pagerank_score"])

print(f"PageRank finished in {time.time() - start_time:.2f}s")

Starting PageRank...


Graph loaded. Nodes: 6301


Iteration 1 completed


Iteration 2 completed


Iteration 3 completed


...

Iteration 8 completed


Iteration 9 completed


Iteration 10 completed


[Stage 100:>                                                        (0 + 4) / 4]

[IO] Saved 20 rows to: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/pagerank_top20.csv
PageRank finished in 41.80s


## 4. Part A — Multi-Source Personalized PageRank

In [6]:
# write some code here
# - parameters: sources list, alpha, iterations, partitions
# - init mass 1/|S| on sources; others 0
# - on jump and dangling mass, teleport uniformly to S
# - use mapPartitions(..., preservesPartitioning=True) when transforming keyed RDDs
# - compute top-20 and write outputs/ppr_top20.csv
# - save any DF stage plan to proof/plan_ppr.txt
import time
from operator import add

SOURCES = ['0', '24', '135', '2054', '1029']
ALPHA_PPR = 0.85
ITERS_PPR = 10

print(f"Starting Personalized PageRank (Sources={SOURCES})...")
start_ppr = time.time()

source_set = set(SOURCES)
sources_bc = sc.broadcast(source_set)
num_sources = len(source_set)

ranks = all_nodes.map(lambda n: (n, 1.0 / num_sources if n in sources_bc.value else 0.0)) \
                 .partitionBy(PARTITIONS) \
                 .cache()

with open(PROOF_DIR / "plan_ppr.txt", "w") as f:
    f.write(ranks.toDebugString().decode("utf-8"))

for i in range(ITERS_PPR):
    dangling_mass = ranks.subtractByKey(links).map(lambda x: x[1]).sum()
    
    teleport_total = (1.0 - ALPHA_PPR) + (ALPHA_PPR * dangling_mass)
    teleport_per_source = teleport_total / num_sources

    contribs = links.join(ranks).flatMap(
        lambda kv: [(target, kv[1][1] / len(kv[1][0])) for target in kv[1][0]]
    )

    def ppr_partition_mapper(iterator):
        local_sources = sources_bc.value
        for node_id, (contrib_wrapper, _) in iterator:
            s = contrib_wrapper or 0.0
            new_val = s * ALPHA_PPR
            if node_id in local_sources:
                new_val += teleport_per_source
            yield (node_id, new_val)

    new_ranks = (
        contribs.reduceByKey(add)
        .rightOuterJoin(ranks)
        .mapPartitions(ppr_partition_mapper, preservesPartitioning=True)
    )

    new_ranks.cache()
    ranks.unpersist()
    ranks = new_ranks
    
    print(f"PPR Iteration {i+1} completed")

top_20_ppr = get_top_k(ranks, 20)
output_ppr = OUTPUTS_DIR / "ppr_top20.csv"
save_results_csv(top_20_ppr, str(output_ppr), ["node_id", "ppr_score"])

print(f"PPR finished in {time.time() - start_ppr:.2f}s")

Starting Personalized PageRank (Sources=['0', '24', '135', '2054', '1029'])...


PPR Iteration 1 completed


PPR Iteration 2 completed


PPR Iteration 3 completed


PPR Iteration 4 completed


...

PPR Iteration 8 completed


PPR Iteration 9 completed


PPR Iteration 10 completed


[Stage 199:>                                                        (0 + 4) / 4]

[IO] Saved 20 rows to: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/ppr_top20.csv
PPR finished in 35.72s


## 5. Part B — TrainSpamClassifier (SGD)

In [7]:
# write some code here
# - parameters: delta, epochs, shuffle flag, numReducers=1
# - read training lines: docid label f1 f2 ...
# - emit (0, (docid, isSpam, features)) and groupByKey(1) to a single learner
# - implement SGD updates on the reducer side; save model to outputs/model_*/part-00000
import math
import random
from collections import defaultdict

DELTA = 0.002
EPOCHS = 10

def train_spam_classifier(input_path, output_path, learning_rate, num_epochs, shuffle=False):
    """Train a spam classifier using SGD on a single reducer (groupByKey)."""
    print(f"--- Training on {input_path.name} (Shuffle={shuffle}) ---")
    start_time = time.time()
    
    def parse_line(line):
        parts = line.split()
        if len(parts) < 2:
            return None
        label = 1.0 if parts[1] == "spam" else 0.0
        features = [int(f) for f in parts[2:]]
        return (label, features)
        
    def sgd_reducer(iterator):
        data = list(iterator)
        weights = defaultdict(float)
        
        for epoch in range(num_epochs):
            if shuffle:
                random.shuffle(data)
            
            for label, features in data:
                score = sum(weights.get(f, 0.0) for f in features)

                if score > 20: prob = 1.0
                elif score < -20: prob = 0.0
                else: prob = 1.0 / (1.0 + math.exp(-score))
                
                # w[f] += (y - p) * delta
                error = label - prob
                if error != 0:
                    update = error * learning_rate
                    for f in features:
                        weights[f] += update
        
        for f, w in sorted(weights.items()):
            yield f"{f}\t{w}"

    model_rdd = (
        sc.textFile(str(input_path))
        .map(parse_line)
        .filter(lambda x: x is not None)
        .map(lambda x: (0, x))    
        .groupByKey(numPartitions=1) 
        .flatMapValues(sgd_reducer)
        .values()
    )

    clear_output(str(output_path))
    model_rdd.saveAsTextFile(str(output_path))
    print(f"Model saved to {output_path} (Time: {time.time() - start_time:.2f}s)")

In [11]:
# Entraînement Group X 
print(">>> Training Model X...")
train_spam_classifier(
    input_path=spam_dir / "spam.train.group_x.txt", 
    output_path=OUTPUTS_DIR / "model_group_x", 
    learning_rate=DELTA, 
    num_epochs=EPOCHS
)

>>> Training Model X...
--- Training on spam.train.group_x.txt (Shuffle=False) ---


Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/model_group_x (Time: 21.91s)


In [12]:
# Entraînement Group Y 
print(">>> Training Model Y...")
train_spam_classifier(
    input_path=spam_dir / "spam.train.group_y.txt", 
    output_path=OUTPUTS_DIR / "model_group_y", 
    learning_rate=DELTA, 
    num_epochs=EPOCHS
)

>>> Training Model Y...
--- Training on spam.train.group_y.txt (Shuffle=False) ---


Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/model_group_y (Time: 19.30s)


In [13]:
# Entraînement Britney 
print(">>> Training Model Britney...")
train_spam_classifier(
    input_path=spam_dir / "spam.train.britney.txt", 
    output_path=OUTPUTS_DIR / "model_britney", 
    learning_rate=DELTA, 
    num_epochs=EPOCHS
)

>>> Training Model Britney...
--- Training on spam.train.britney.txt (Shuffle=False) ---


Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/model_britney (Time: 841.43s)


## 6. Part B — ApplySpamClassifier

In [14]:
# write some code here
# - load model tuple file to dict or broadcast
# - score test instances and emit (docid, score, predicted_label)
# - write outputs/predictions_*/
def apply_spam_classifier(model_path, data_path, output_path):
    """
    Applique un modèle sauvegardé sur un dataset et génère des prédictions.
    Sortie : docid, score, predicted_label
    """
    print(f"--- Predicting on {data_path.name} using {model_path.name} ---")
    t0 = time.time()
    clear_output(str(output_path))

    raw_model = sc.textFile(str(model_path / "part-00000"))
    model_map = raw_model.map(lambda line: line.split('\t')) \
                         .map(lambda p: (int(p[0]), float(p[1]))) \
                         .collectAsMap()
    
    weights_bc = sc.broadcast(model_map)
    print(f"   Model loaded: {len(model_map)} features broadcasted.")

    def parse_for_prediction(line):
        parts = line.split()
        if len(parts) < 2: return None
        docid = parts[0]
        features = [int(f) for f in parts[2:]]
        return (docid, features)

    def predict(item):
        docid, features = item
        local_weights = weights_bc.value
        
        # Score = somme des poids des features présentes
        score = sum(local_weights.get(f, 0.0) for f in features)
        
        label = "spam" if score > 0 else "ham"
        return f"{docid},{score:.5f},{label}"

    predictions = (
        sc.textFile(str(data_path))
        .map(parse_for_prediction)
        .filter(lambda x: x is not None)
        .map(predict)
    )

    # Save
    predictions.saveAsTextFile(str(output_path))
    
    weights_bc.unpersist()
    print(f"   Predictions saved to {output_path.name} (Time: {time.time() - t0:.2f}s)")

In [16]:
apply_spam_classifier(
    model_path=OUTPUTS_DIR / "model_group_x",
    data_path=spam_dir / "spam.train.group_x.txt",
    output_path=OUTPUTS_DIR / "predictions_group_x"
)

--- Predicting on spam.train.group_x.txt using model_group_x ---


   Model loaded: 296775 features broadcasted.


   Predictions saved to predictions_group_x (Time: 4.96s)


In [17]:
apply_spam_classifier(
    model_path=OUTPUTS_DIR / "model_britney",
    data_path=spam_dir / "spam.train.group_x.txt",
    output_path=OUTPUTS_DIR / "predictions_britney_on_x"
)

--- Predicting on spam.train.group_x.txt using model_britney ---


   Model loaded: 966051 features broadcasted.


   Predictions saved to predictions_britney_on_x (Time: 5.70s)


## 7. Part B — ApplyEnsembleSpamClassifier

In [18]:
# write some code here
# - --method average or vote
# - load multiple part-00000 model files; broadcast
# - average scores or majority vote; write outputs and a small sample

def apply_ensemble_classifier(model_paths, data_path, output_path, method="average"):
    """
    Combine plusieurs modèles pour prédire.
    method: "average" (moyenne des scores) ou "vote" (majorité).
    """
    print(f"--- Ensemble Prediction ({method}) on {data_path.name} ---")
    t0 = time.time()
    clear_output(str(output_path))

    # Charger les modèles
    models_list = []
    for path in model_paths:
        raw = sc.textFile(str(path / "part-00000"))
        w_map = raw.map(lambda l: l.split('\t')) \
                   .map(lambda p: (int(p[0]), float(p[1]))) \
                   .collectAsMap()
        models_list.append(w_map)
    
    models_bc = sc.broadcast(models_list)
    print(f"   Loaded {len(models_list)} models for ensemble.")

    def parse_data(line):
        parts = line.split()
        if len(parts) < 2: return None
        return (parts[0], [int(f) for f in parts[2:]])

    def predict_ensemble(item):
        docid, features = item
        all_models = models_bc.value

        scores = []
        for w in all_models:
            s = sum(w.get(f, 0.0) for f in features)
            scores.append(s)
        
        final_score = 0.0
        
        if method == "average":
            final_score = sum(scores) / len(scores)
        
        elif method == "vote":
            votes = sum(1 if s > 0 else -1 for s in scores)
            final_score = float(votes) 
            
        label = "spam" if final_score > 0 else "ham"
        return f"{docid},{final_score:.5f},{label}"

    (
        sc.textFile(str(data_path))
        .map(parse_data)
        .filter(lambda x: x is not None)
        .map(predict_ensemble)
        .saveAsTextFile(str(output_path))
    )
    
    models_bc.unpersist()
    print(f"   Ensemble ({method}) saved to {output_path.name} (Time: {time.time() - t0:.2f}s)")

In [20]:
models_to_combine = [OUTPUTS_DIR / "model_group_x", OUTPUTS_DIR / "model_group_y"]

apply_ensemble_classifier(
    model_paths=models_to_combine,
    data_path=spam_dir / "spam.train.group_x.txt",
    output_path=OUTPUTS_DIR / "predictions_all",
    method="vote"
)

--- 🤝 Ensemble Prediction (vote) on spam.train.group_x.txt ---


   Loaded 2 models for ensemble.


   Ensemble (vote) saved to predictions_all (Time: 3.81s)


## 8. Evaluation and shuffle study

In [24]:
# write some code here
# - compute ROC-AUC with Spark ML if desired
# - or invoke external compute_spam_metrics if available (optional)
# - implement --shuffle: random key + sortBy to permute training before SGD
# - run 10 trials on britney; summarize in outputs/metrics.md
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.types import StructType, StructField, DoubleType

def evaluate_auc(prediction_path, label_file=spam_dir / "spam.test.qrels.txt"):
    """
    Calcule le score ROC-AUC en comparant les prédictions aux labels de référence.
    """
    def parse_qrels(line):
        parts = line.split()
        if len(parts) < 2: return None
        label = 1.0 if parts[1] == "spam" or parts[1] == "1" else 0.0
        return (parts[0], label)

    labels_rdd = sc.textFile(str(label_file)).map(parse_qrels).filter(lambda x: x is not None)

    def parse_preds(line):
        parts = line.split(',')
        if len(parts) < 2: return None
        return (parts[0], float(parts[1])) # (docid, score)

    preds_rdd = sc.textFile(str(prediction_path)).map(parse_preds).filter(lambda x: x is not None)

    joined = preds_rdd.join(labels_rdd) 

    if joined.isEmpty():
        print("   AVERTISSEMENT: Aucune correspondance trouvée entre prédictions et qrels.")
        print("      Vérifiez que les DocIDs correspondent.")
        return 0.0

    data = joined.map(lambda x: (x[1][0], x[1][1]))
    schema = StructType([
        StructField("rawPrediction", DoubleType(), False),
        StructField("label", DoubleType(), False)
    ])
    df = spark.createDataFrame(data, schema)

    evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
    auc = evaluator.evaluate(df)
    
    print(f"   ROC-AUC Score: {auc:.5f}")
    return auc

# --- Exécution des tests ---
print("\n--- Résultats Finaux ---")

training_labels = spam_dir / "spam.train.group_x.txt"

print("1. Model X sur ses propres données ")
auc_x = evaluate_auc(OUTPUTS_DIR / "predictions_group_x", label_file=training_labels)

print("2. Model Britney sur données X ")
auc_britney = evaluate_auc(OUTPUTS_DIR / "predictions_britney_on_x", label_file=training_labels)

print("3. Ensemble Vote (X+Y) sur données X")
auc_ensemble = evaluate_auc(OUTPUTS_DIR / "predictions_all", label_file=training_labels)


--- Résultats Finaux ---
1. Model X sur ses propres données (Doit être bon)


   ROC-AUC Score: 0.99651
2. Model Britney sur données X (Doit être très bon)


   ROC-AUC Score: 0.76866
3. Ensemble Vote (X+Y) sur données X
   ROC-AUC Score: 0.86781


In [28]:
import statistics
import shutil
import os
import time

def run_shuffle_study(target_dataset, test_dataset, num_trials=5):
    """
    Lance N entraînements avec mélange aléatoire (shuffle=True)
    pour voir la stabilité du SGD.
    """
    print(f"\n--- Starting Shuffle Study ({num_trials} trials) ---")
    print(f"Training on: {target_dataset.name}")
    
    scores = []
    
    for i in range(num_trials):
        print(f"   Trial {i+1}/{num_trials}...")
        trial_model_path = OUTPUTS_DIR / f"shuffle_model_{i}"
        train_spam_classifier(
            input_path=target_dataset,
            output_path=trial_model_path,
            learning_rate=DELTA,
            num_epochs=5, 
            shuffle=True 
        )
        
        trial_pred_path = OUTPUTS_DIR / f"shuffle_pred_{i}"
        apply_spam_classifier(trial_model_path, test_dataset, trial_pred_path)

        auc = evaluate_auc(trial_pred_path, label_file=test_dataset)
        scores.append(auc)

        clear_output(str(trial_model_path))
        clear_output(str(trial_pred_path))

    mean_score = statistics.mean(scores)
    stdev_score = statistics.stdev(scores) if len(scores) > 1 else 0.0
    
    print(f"\n--- Study Results ---")
    print(f"Scores: {scores}")
    print(f"Mean AUC: {mean_score:.5f}")
    print(f"Std Dev:  {stdev_score:.5f}")
    
    return scores, mean_score, stdev_score

study_scores, mean_val, std_val = run_shuffle_study(
    target_dataset=spam_dir / "spam.train.group_y.txt", 
    test_dataset=spam_dir / "spam.train.group_y.txt",
    num_trials=10 
)


metrics_content = f"""
# Big Data Analytics - Assignment 03 Metrics
**Author:** Badr TAJINI
**Date:** {time.strftime("%Y-%m-%d %H:%M:%S")}

## 1. Spam Classification Performance (ROC-AUC)
* **Model Group X (Self-Test):** {auc_x:.5f}
* **Model Britney (Generalization on X):** {auc_britney:.5f}
* **Ensemble Vote (X+Y on X):** {auc_ensemble:.5f}

## 2. Shuffle Study (Stability Analysis)
* **Dataset Used:** {spam_dir / "spam.train.group_y.txt"}
* **Trials:** {len(study_scores)}
* **Mean AUC:** {mean_val:.5f}
* **Std Deviation:** {std_val:.5f}
* **Raw Scores:** {study_scores}

> *Analysis: A low standard deviation indicates that the SGD algorithm is robust to the order of input data.*
"""

with open(OUTPUTS_DIR / "metrics.md", "w") as f:
    f.write(metrics_content)

print(f"\nRapport final généré : {OUTPUTS_DIR / 'metrics.md'}")


--- Starting Shuffle Study (10 trials) ---
Training on: spam.train.group_y.txt
   Trial 1/10...
--- Training on spam.train.group_y.txt (Shuffle=True) ---


Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_0 (Time: 8.67s)
--- Predicting on spam.train.group_y.txt using shuffle_model_0 ---
   Model loaded: 236865 features broadcasted.


   Predictions saved to shuffle_pred_0 (Time: 3.22s)
   ROC-AUC Score: 0.99890
Deleted old output: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_0
Deleted old output: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_pred_0
   Trial 2/10...
--- Training on spam.train.group_y.txt (Shuffle=True) ---


Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_1 (Time: 11.04s)
--- Predicting on spam.train.group_y.txt using shuffle_model_1 ---


   Model loaded: 236865 features broadcasted.


...

Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_8 (Time: 11.16s)
--- Predicting on spam.train.group_y.txt using shuffle_model_8 ---
   Model loaded: 236865 features broadcasted.


   Predictions saved to shuffle_pred_8 (Time: 2.87s)
   ROC-AUC Score: 0.99984
Deleted old output: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_8
Deleted old output: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_pred_8
   Trial 10/10...
--- Training on spam.train.group_y.txt (Shuffle=True) ---


Model saved to /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_9 (Time: 11.36s)
--- Predicting on spam.train.group_y.txt using shuffle_model_9 ---


   Model loaded: 236865 features broadcasted.


   Predictions saved to shuffle_pred_9 (Time: 3.31s)
   ROC-AUC Score: 0.99951
Deleted old output: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_model_9
Deleted old output: /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/shuffle_pred_9

--- Study Results ---
Scores: [0.9988969318494189, 0.9990449473840016, 0.9995348083198827, 0.9998061701332845, 0.9992775432240604, 0.9998942746181552, 0.9997568316217568, 0.9995030907053294, 0.9998414119272329, 0.999506614884724]
Mean AUC: 0.99951
Std Dev:  0.00034

Rapport final généré : /mnt/c/Users/ellio/OneDrive/Documents/Postbac/E5-DSIA/elliot/big data/bigdata/lab3/assignment/outputs/metrics.md


## 9. Spark UI evidence
Open http://localhost:4040 during runs. Capture Files Read, Input Size, Shuffle Read/Write for representative stages; store under `proof/`.

## 10. Environment and reproducibility

In [ ]:
# write some code here
# - print Java version, Spark conf, OS info
# - save ENV.md: versions + key configs
import json
import subprocess

def get_java_version():
    try:
        output = subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT)
        return output.decode("utf-8").strip().splitlines()[0]
    except Exception as exc:
        return f"Unavailable ({exc})"

java_output = get_java_version()
print(f"Java: {java_output}")

print("Spark configuration (selected):")
conf_items = sorted(spark.sparkContext.getConf().getAll())
for key, value in conf_items:
    print(f" - {key} = {value}")

env_summary = {
    "python": sys.version,
    "spark": spark.version,
    "pyspark": pyspark.__version__,
    "java": java_output,
    "os": platform.platform(),
    "spark_conf": {k: v for k, v in conf_items if k.startswith("spark.")}}

env_lines = [
    "# Environment Summary",
    "",
    f"- Python: {sys.version.split()[0]}",
    f"- Spark: {spark.version}",
    f"- PySpark: {pyspark.__version__}",
    f"- Java: {java_output}",
    f"- OS: {platform.platform()}",
    "",
    "## Spark Configuration"]

env_lines.extend(f"- {k} = {v}" for k, v in env_summary["spark_conf"].items())

ENV_PATH = Path("ENV.md")
ENV_PATH.write_text("\n".join(env_lines) + "\n")

print(f"Environment details saved to {ENV_PATH.resolve()}")